# Module 2: RAG (Retrieval-Augmented Generation) System

**Duration:** ~45 minutes

**Environment:** AWS g4dn.4xlarge (or local GPU)

## Learning Objectives
- Understand how RAG enhances LLM capabilities
- Process documents and create embeddings
- Build a ChromaDB vector store
- Implement a complete RAG pipeline with LangChain
- Compare RAG-enhanced responses with baseline

## Overview
RAG allows us to augment an LLM with external knowledge without retraining. We'll build a system that retrieves relevant F5 documentation and includes it as context when generating responses.

## Prerequisites
- Completed Module 1 (or baseline_results.json exists)
- Virtual environment activated with dependencies installed

## 2.1 Set Working Directory

First, we ensure we're running from the correct project directory.

In [ ]:
import os

def find_project_root():
    """Find the project root directory by looking for key files."""
    current = os.getcwd()
    
    if os.path.exists(os.path.join(current, 'data', 'f5_docs')):
        return current
    
    parent = os.path.dirname(current)
    grandparent = os.path.dirname(parent)
    
    if os.path.exists(os.path.join(grandparent, 'data', 'f5_docs')):
        return grandparent
    
    home = os.path.expanduser('~')
    common_paths = [
        os.path.join(home, 'llm-finetuning-rag-lab'),
        os.path.join(home, 'LLM-FineTuning-RAG-Lab'),
        '/root/llm-finetuning-rag-lab',
    ]
    
    for path in common_paths:
        if os.path.exists(os.path.join(path, 'data', 'f5_docs')):
            return path
    
    return None

PROJECT_ROOT = find_project_root()

if PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)
    print(f"Working directory set to: {PROJECT_ROOT}")
else:
    print("ERROR: Could not find project root directory!")
    print("Please run: cd ~/llm-finetuning-rag-lab && source venv/bin/activate && jupyter lab")
    raise FileNotFoundError("Project root not found.")

## 2.2 Environment Setup

In [ ]:
import torch

print("Environment Check")
print("=" * 50)
print(f"Working directory: {os.getcwd()}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Verify data exists
if os.path.exists('data/f5_docs'):
    print(f"\nF5 docs directory found")
    for f in os.listdir('data/f5_docs'):
        print(f"  - {f}")
else:
    print("Data directory not found")

In [ ]:
# Verify RAG packages
import langchain
import chromadb
import sentence_transformers

print("Package Versions")
print("=" * 50)
print(f"langchain: {langchain.__version__}")
print(f"chromadb: {chromadb.__version__}")
print(f"sentence-transformers: {sentence_transformers.__version__}")

## 2.3 Load the Base Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"Loading {MODEL_NAME}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

print(f"Model loaded on {model.device}")

## 2.4 Understanding RAG Architecture

RAG consists of three main stages:

1. **Indexing**: Convert documents to embeddings and store in vector database
2. **Retrieval**: Find relevant documents for a given query
3. **Generation**: Use retrieved context to generate informed responses

```
┌─────────────┐     ┌─────────────┐     ┌─────────────┐
│  Documents  │────▶│  Embeddings │────▶│  Vector DB  │
└─────────────┘     └─────────────┘     └──────┬──────┘
                                               │
┌─────────────┐     ┌─────────────┐           │
│   Query     │────▶│  Retrieval  │◀──────────┘
└─────────────┘     └──────┬──────┘
                           │
                    ┌──────▼──────┐     ┌─────────────┐
                    │   Context   │────▶│     LLM     │────▶ Response
                    └─────────────┘     └─────────────┘
```

## 2.5 Load F5 Documentation

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# Load all text files from the F5 docs directory
loader = DirectoryLoader(
    "data/f5_docs/",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

documents = loader.load()

print(f"Loaded {len(documents)} documents")
for doc in documents:
    source = doc.metadata.get('source', 'unknown')
    chars = len(doc.page_content)
    print(f"  - {source}: {chars:,} characters")

In [ ]:
# Preview a document
print("Sample document content (first 500 chars):")
print("-" * 50)
print(documents[0].page_content[:500])

## 2.6 Split Documents into Chunks

In [ ]:
# =============================================================================
# WHY SPLIT DOCUMENTS INTO CHUNKS?
# =============================================================================
# LLMs have limited context windows. We split documents into smaller chunks so we can:
#   1. Retrieve only the most relevant portions (not entire documents)
#   2. Fit multiple relevant chunks into the context window
#   3. Get more precise semantic matching during retrieval
#
# CHUNK SIZE (500 chars): Balance between context and precision
#   - Too small: Loses context, fragments sentences
#   - Too large: Less precise retrieval, wastes context window space
#
# CHUNK OVERLAP (50 chars): Prevents losing information at boundaries
# =============================================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print(f"Split into {len(chunks)} chunks")
print(f"Average chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} characters")

In [ ]:
# Preview some chunks
print("Sample chunks:")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i+1} ({len(chunk.page_content)} chars) ---")
    print(chunk.page_content[:200] + "...")

## 2.7 Create Embeddings

In [ ]:
# =============================================================================
# WHY THIS EMBEDDING MODEL?
# =============================================================================
# Embeddings convert text into dense numerical vectors that capture semantic meaning.
# Similar texts have similar vectors (close in vector space).
#
# all-MiniLM-L6-v2 is chosen because:
#   1. FAST: Small model (80MB) with quick inference
#   2. FREE: No API keys or costs - runs locally
#   3. GOOD QUALITY: Trained on 1B+ sentence pairs
#   4. 384 DIMENSIONS: Good balance of precision vs. storage
# =============================================================================

from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

print(f"Loading embedding model: {EMBEDDING_MODEL}")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Embedding model loaded")

In [ ]:
# Test the embeddings
test_text = "F5 BIG-IP virtual server configuration"
test_embedding = embeddings.embed_query(test_text)

print(f"Embedding dimension: {len(test_embedding)}")
print(f"First 10 values: {test_embedding[:10]}")

## 2.8 Create Vector Store

In [ ]:
# =============================================================================
# WHY CHROMADB?
# =============================================================================
# Vector databases store embeddings and enable fast similarity search.
# ChromaDB is chosen because:
#   1. RUNS LOCALLY: No external service or API keys needed
#   2. SIMPLE: Easy to set up and use
#   3. PERSISTENT: Can save to disk and reload later
# =============================================================================

from langchain_community.vectorstores import Chroma

print("Creating vector store...")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="f5_docs",
    persist_directory="./chroma_db"
)

print(f"Vector store created with {len(chunks)} chunks")

## 2.9 Test Retrieval

In [ ]:
# Create a retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# Test query
test_query = "How do I configure SSL offloading?"
print(f"Query: {test_query}\n")

retrieved_docs = retriever.invoke(test_query)

print(f"Retrieved {len(retrieved_docs)} documents:\n")
for i, doc in enumerate(retrieved_docs, 1):
    source = doc.metadata.get('source', 'unknown').split('/')[-1]
    print(f"--- Document {i} (from {source}) ---")
    print(doc.page_content[:300] + "...\n")

## 2.10 Build the RAG Pipeline

In [ ]:
from transformers import pipeline

# Create a text generation pipeline
text_gen_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
    return_full_text=False
)

print("LLM pipeline created")

In [ ]:
# Create a prompt template that includes context
RAG_TEMPLATE = """<|system|>
You are an F5 Networks technical expert. Use the following context to answer the question accurately. If the answer is not in the context, say so but still provide helpful information.</s>
<|user|>
Context:
{context}

Question: {question}</s>
<|assistant|>
"""

print("Prompt template created")

In [ ]:
# =============================================================================
# THE RAG PIPELINE: How it all works together
# =============================================================================
#   User Question
#        ↓
#   [1. RETRIEVE] → Search vector DB for relevant chunks
#        ↓
#   [2. AUGMENT]  → Add retrieved context to the prompt
#        ↓
#   [3. GENERATE] → LLM generates answer using the context
#        ↓
#   Answer + Sources
# =============================================================================

def query_rag(question):
    """
    Query the RAG system and return the response with sources.
    """
    # Step 1: RETRIEVE - Find relevant chunks
    retrieved_docs = retriever.invoke(question)
    
    # Step 2: AUGMENT - Build context from chunks
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)
    
    # Step 3: Create full prompt with context
    full_prompt = RAG_TEMPLATE.format(context=context, question=question)
    
    # Step 4: GENERATE - Get LLM response
    response = text_gen_pipeline(full_prompt)[0]["generated_text"]
    
    # Extract sources for attribution
    sources = [doc.metadata.get('source', 'unknown').split('/')[-1] 
               for doc in retrieved_docs]
    
    return response, list(set(sources))

print("RAG query function ready")

## 2.11 Test RAG System

In [ ]:
# Test with a single question
test_question = "How do I configure SSL offloading on F5 BIG-IP?"

print(f"Question: {test_question}\n")
print("Querying RAG system...\n")

response, sources = query_rag(test_question)

print(f"Response:\n{response}\n")
print(f"Sources: {', '.join(set(sources))}")

## 2.12 Evaluate RAG on All Questions

In [ ]:
# Same evaluation questions from Module 1
eval_questions = [
    "What is a virtual server in F5 BIG-IP and what is its purpose?",
    "How do I configure SSL offloading on F5 BIG-IP?",
    "Write an iRule to redirect HTTP to HTTPS.",
    "What is the difference between Least Connections and Round Robin load balancing?",
    "How do I troubleshoot a pool member that is marked as offline?"
]

print(f"Evaluating {len(eval_questions)} questions with RAG...\n")
print("=" * 80)

In [ ]:
# Store RAG responses
rag_responses = []

for i, question in enumerate(eval_questions, 1):
    print(f"\n{'='*80}")
    print(f"Question {i}: {question}")
    print("-" * 80)
    
    response, sources = query_rag(question)
    rag_responses.append({
        "question": question,
        "response": response,
        "sources": list(set(sources))
    })
    
    print(f"Response:\n{response}")
    print(f"\nSources: {', '.join(set(sources))}")

print(f"\n{'='*80}")
print("RAG evaluation complete!")

## 2.13 Compare with Baseline

In [ ]:
import json

# Load baseline results
try:
    with open("results/baseline_results.json", "r") as f:
        baseline_data = json.load(f)
    baseline_responses = baseline_data["responses"]
    print("Baseline results loaded")
except FileNotFoundError:
    print("Baseline results not found. Run Module 1 first.")
    baseline_responses = []

In [ ]:
# Side-by-side comparison
if baseline_responses:
    print("SIDE-BY-SIDE COMPARISON")
    print("=" * 80)
    
    for i, (baseline, rag) in enumerate(zip(baseline_responses, rag_responses), 1):
        print(f"\n{'='*80}")
        print(f"Question {i}: {baseline['question']}")
        
        print(f"\n{'─'*40}")
        print("BASELINE RESPONSE:")
        print(baseline['response'][:400] + "..." if len(baseline['response']) > 400 else baseline['response'])
        
        print(f"\n{'─'*40}")
        print("RAG RESPONSE:")
        print(rag['response'][:400] + "..." if len(rag['response']) > 400 else rag['response'])
        print(f"   Sources: {', '.join(rag['sources'])}")

## 2.14 Save RAG Results

In [ ]:
os.makedirs("results", exist_ok=True)

results = {
    "model": MODEL_NAME,
    "type": "rag",
    "embedding_model": EMBEDDING_MODEL,
    "chunk_size": 500,
    "responses": rag_responses
}

with open("results/rag_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("RAG results saved to results/rag_results.json")

## 2.15 Understanding RAG Trade-offs

### Advantages of RAG
- **No training required** - Just add documents  
- **Easy to update** - Add/remove documents anytime  
- **Source attribution** - Know where answers come from  
- **Current information** - Not limited by training cutoff  

### Limitations of RAG
- **Retrieval quality matters** - Bad retrieval = bad answers  
- **Context window limits** - Can only use limited context  
- **Latency overhead** - Retrieval adds time  
- **Doesn't learn patterns** - Can't generalize beyond documents

## 2.16 Summary

In this module, we:

1. Built a document processing pipeline
2. Created embeddings using sentence-transformers
3. Stored vectors in ChromaDB
4. Implemented a complete RAG chain
5. Compared RAG responses with baseline

### Next Steps
In **Module 3**, we'll fine-tune the model itself on F5-specific data using QLoRA.

In [ ]:
# Cleanup
import gc
del model
del tokenizer
del vectorstore
del embeddings
del text_gen_pipeline
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared - ready for Module 3")